In [5]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Sequence
import time

# Core LangGraph components, including checkpoints
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver

# LLM and messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

# Load API keys and set up tracing
load_dotenv()
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Intro to LangGraph"

# -------------------------------
# Define the State
# -------------------------------
class GraphState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]


# -------------------------------
# Define Nodes
# -------------------------------
def call_llm(state: GraphState):
    print("--- Calling LLM ---")
    llm = ChatOpenAI(model="gpt-4o")
    response = llm.invoke(state['messages'])
    return {"messages": [response]}


def wait_for_a_moment(state: GraphState):
    print("--- Taking a short break... ---")
    time.sleep(2)
    return {}


# -------------------------------
# Build the Graph
# -------------------------------
workflow = StateGraph(GraphState)
workflow.add_node("llm", call_llm)
workflow.add_node("pause", wait_for_a_moment)

workflow.add_edge(START, "llm")
workflow.add_edge("llm", "pause")
workflow.add_edge("pause", END)

# -------------------------------
# Set up Memory (checkpointer)
# -------------------------------
# ✅ Use context manager instead of .setup()
with SqliteSaver.from_conn_string(":memory:") as memory:
    app = workflow.compile(
        checkpointer=memory,
        interrupt_before=["pause"]
    )

    # --- Run the Graph with Breakpoints ---
    config = {"configurable": {"thread_id": "breakpoint-thread-1"}}

    print("--- Starting graph execution ---")
    initial_state = app.invoke({"messages": [HumanMessage(content="Hello!")]}, config)

    print("\n--- Graph Paused ---")
    print("Current state (at breakpoint):")
    for msg in initial_state['messages']:
        print(f"- {msg.type}: {msg.content}")

    print("\n--- Resuming graph execution ---")
    final_state = app.invoke(None, config)

    print("\n--- Graph Finished ---")
    print("Final state:")
    for msg in final_state['messages']:
        print(f"- {msg.type}: {msg.content}")


--- Starting graph execution ---
--- Calling LLM ---

--- Graph Paused ---
Current state (at breakpoint):
- human: Hello!
- ai: Hi there! How can I assist you today?

--- Resuming graph execution ---
--- Taking a short break... ---

--- Graph Finished ---
Final state:
- human: Hello!
- ai: Hi there! How can I assist you today?
